# 04_baseline_models.ipynb

Classical Machine Learning Baseline Models Evaluation & Benchmarking Notebook.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.utils.config import load_config
from src.utils.reproducibility import set_seed
from src.data.loaders import CICIDS2017Loader, DatasetNotFoundError
from src.data.dataset_adapters import CICIDS2017Adapter
from src.preprocessing.pipeline import CyberthreatPreprocessingPipeline
from src.models.model_factory import ModelFactory
from src.evaluation.experiments import ExperimentRunner

set_seed(42)
config = load_config("../configs/config.yaml")
print("Initialized Baseline Models Evaluation Environment.")


In [ ]:
# 1. Dataset Ingestion & Preprocessing
try:
    loader = CICIDS2017Loader("../data/raw/CICIDS2017")
    df_raw = loader.load_merged_dataset(sample_frac=0.1)
except DatasetNotFoundError:
    print("[NOTE] Raw dataset absent. Using synthetic benchmark dataset...")
    df_raw = pd.DataFrame({
        " Destination Port": np.random.choice([80, 443, 22, 8080], 1000),
        " Flow Duration": np.random.exponential(1000, 1000),
        " Total Fwd Packets": np.random.randint(1, 50, 1000),
        " Total Backward Packets": np.random.randint(0, 50, 1000),
        " Total Length of Fwd Packets": np.random.uniform(10, 5000, 1000),
        " Total Length of Bwd Packets": np.random.uniform(0, 5000, 1000),
        " Flow Bytes/s": np.random.uniform(0, 1e6, 1000),
        " Label": np.random.choice(["BENIGN", "DDoS", "PortScan"], 1000, p=[0.7, 0.2, 0.1])
    })

adapter = CICIDS2017Adapter()
X, y_bin, y_multi = adapter.extract_labels(df_raw)

pipeline = CyberthreatPreprocessingPipeline(random_state=config.system.seed)
processed = pipeline.fit_transform_splits(X, y_bin, y_multi)

X_train, y_train = processed["X_train"], processed["y_train_binary"]
X_test, y_test = processed["X_test"], processed["y_test_binary"]

print(f"Data Splits: Train={X_train.shape}, Test={X_test.shape}")


In [ ]:
# 2. Instantiate Classical Baseline Models
models = {
    "LogisticRegression": ModelFactory.create_model("logistic_regression", config.models.baselines.logistic_regression),
    "RandomForest": ModelFactory.create_model("random_forest", config.models.baselines.random_forest),
    "DecisionTree": ModelFactory.create_model("decision_tree", {"max_depth": 15}),
    "SVM": ModelFactory.create_model("svm", config.models.baselines.svm),
}

print(f"Instantiated {len(models)} baseline models for benchmarking.")


In [ ]:
# 3. Execute Experiment & Generate Metrics Table
runner = ExperimentRunner(results_dir="../results")
df_summary = runner.run_baseline_comparison(
    models, X_train, y_train, X_test, y_test, experiment_name="04_baseline_models"
)

print("
=== Baseline Classifier Comparative Metrics Table ===")
print(df_summary.to_string(index=False))
